# USGS Water — multi-parameter and instantaneous values

Pull two parameters at once (discharge + gage height), then a sub-daily
(`instantaneous`) discharge series, for the same gauge. USGS Water is a
*tabular* backend, so `download()` returns a `DataFrame` (and writes a CSV
alongside it).

## Setup

A single import of the unified `EarthLens` entry point. Each request below
shares the same gauge (`01646500`, the Potomac at Little Falls) and
bounding box.

In [ ]:
from earthlens.earthlens import EarthLens

## 1 · Two parameters at once (daily)

Ask for `discharge` and `gage_height` together over a two-week window. We
build the request first so the source, parameters, dates, and `service`
are easy to read on their own.

In [ ]:
daily_request = EarthLens(
    data_source="usgs-water",
    variables=["discharge", "gage_height"],
    start="2023-06-01",
    end="2023-06-15",
    aoi=[-77.2, 38.9, -77.0, 39.0],
    path="_nb_out",
    service="daily",
    sites="01646500",
    api="legacy",
)

### Download the daily table

`download()` fetches the rows into a `DataFrame` and writes the CSV to
`_nb_out`. We keep it on its own line so the request and the fetch read as
two distinct steps.

In [ ]:
daily = daily_request.download(progress_bar=False)

### Summarise per parameter

Grouping by `parameter_code` gives a quick statistical summary of each
series (discharge in ft³/s, gage height in ft).

In [ ]:
daily.groupby("parameter_code")["value"].describe()

## 2 · Instantaneous (sub-daily) values

Switch `service` to `instantaneous` for the same gauge over a three-day
window. As before, the request is built on its own first.

In [ ]:
iv_request = EarthLens(
    data_source="usgs-water",
    variables=["discharge"],
    start="2023-06-01",
    end="2023-06-03",
    aoi=[-77.2, 38.9, -77.0, 39.0],
    path="_nb_out",
    service="instantaneous",
    sites="01646500",
    api="legacy",
)

### Download the instantaneous series

Fetch the sub-daily readings and report how many came back — the gauge
reports every 15 minutes, so a three-day window yields a few hundred rows.

In [ ]:
iv = iv_request.download(progress_bar=False)
print(len(iv), "sub-daily readings")

### Peek at the readings

The first few rows show the per-timestamp discharge values with their
units and quality qualifiers.

In [ ]:
iv.head()